In [ ]:
import json
import os
import time
from datetime import datetime, timezone, timedelta

import pandas as pd
import numpy as np

In [9]:
# Contract parameters
CONTRACT_START = datetime(2026, 4, 6, tzinfo=timezone.utc)
CONTRACT_END   = datetime(2026, 4, 13, tzinfo=timezone.utc)
CONTRACT_DAYS  = (CONTRACT_END - CONTRACT_START).days   # 7
TARGET_MAG     = 6.5
EVENT_SLUG     = "how-many-6pt5-or-above-earthquakes-april-6-12"

# Historical data parameters
HISTORY_YEARS  = 10
MIN_MAG_FETCH  = 4.5
NOW            = datetime(2026, 4, 10, 12, 0, tzinfo=timezone.utc)  # today per CLAUDE.md
END_DATE       = NOW.strftime("%Y-%m-%d")
START_DATE     = NOW.replace(year=NOW.year - HISTORY_YEARS).strftime("%Y-%m-%d")
TOTAL_DAYS     = (NOW - NOW.replace(year=NOW.year - HISTORY_YEARS)).days

MAGNITUDES   = [4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0]
TIME_WINDOWS = [1, 2, 3, 4, 5, 6, 7]

USGS_QUERY_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
GAMMA_API_URL  = "https://gamma-api.polymarket.com/events"
CLOB_URL       = "https://clob.polymarket.com/prices-history"
CACHE_FILE     = "usgs_events.parquet"

print(f"Observation window : {START_DATE}  to  {END_DATE}  ({TOTAL_DAYS} days)")
print(f"Contract window    : {CONTRACT_START.date()}  to  {(CONTRACT_END - timedelta(days=1)).date()}  ({CONTRACT_DAYS} days)")
print(f"Today (simulated)  : {NOW.date()}")

Observation window : 2016-04-10  to  2026-04-10  (3652 days)
Contract window    : 2026-04-06  to  2026-04-12  (7 days)
Today (simulated)  : 2026-04-10


In [10]:
def fetch_all_events(min_mag, start, end):
    """Fetch all earthquakes >= min_mag between start and end. Paginates yearly."""
    start_dt = datetime.fromisoformat(start).replace(tzinfo=timezone.utc)
    end_dt   = datetime.fromisoformat(end).replace(tzinfo=timezone.utc)
    rows = []
    chunk_start = start_dt
    while chunk_start < end_dt:
        chunk_end = min(chunk_start.replace(year=chunk_start.year + 1), end_dt)
        params = (
            f"format=geojson"
            f"&starttime={chunk_start.strftime('%Y-%m-%d')}"
            f"&endtime={chunk_end.strftime('%Y-%m-%d')}"
            f"&minmagnitude={min_mag}"
            f"&orderby=time-asc"
            f"&limit=20000"
        )
        req = urllib.request.Request(
            f"{USGS_QUERY_URL}?{params}", headers={"User-Agent": "Mozilla/5.0"}
        )
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = json.load(resp)
        for feat in data["features"]:
            p = feat["properties"]
            rows.append({
                "time":      datetime.fromtimestamp(p["time"] / 1000, tz=timezone.utc),
                "magnitude": p["mag"],
            })
        print(f"  {chunk_start.year}: {len(data['features'])} events")
        chunk_start = chunk_end
        time.sleep(0.5)
    df = pd.DataFrame(rows)
    df.sort_values("time", inplace=True, ignore_index=True)
    return df


OLD_CACHE = os.path.join("old", "usgs_events.parquet")

if os.path.exists(CACHE_FILE):
    events = pd.read_parquet(CACHE_FILE)
    events["time"] = pd.to_datetime(events["time"], utc=True)
    print(f"Loaded {len(events):,} events from cache: {CACHE_FILE}")
elif os.path.exists(OLD_CACHE):
    events = pd.read_parquet(OLD_CACHE)
    events["time"] = pd.to_datetime(events["time"], utc=True)
    s = datetime.fromisoformat(START_DATE).replace(tzinfo=timezone.utc)
    e = datetime.fromisoformat(END_DATE).replace(tzinfo=timezone.utc)
    events = events[(events["time"] >= s) & (events["time"] <= e)].copy()
    events.reset_index(drop=True, inplace=True)
    events.to_parquet(CACHE_FILE, index=False)
    print(f"Re-used old cache: {len(events):,} events saved to {CACHE_FILE}")
else:
    print(f"Fetching M{MIN_MAG_FETCH}+ events {START_DATE} to {END_DATE} ...")
    events = fetch_all_events(MIN_MAG_FETCH, START_DATE, END_DATE)
    events.to_parquet(CACHE_FILE, index=False)
    print(f"Total: {len(events):,} events saved to {CACHE_FILE}")

print(f"\nDataset: {len(events):,} events  |  "
      f"mag range {events['magnitude'].min():.1f} to {events['magnitude'].max():.1f}")
events.head()

Loaded 74,601 events from cache: usgs_events.parquet

Dataset: 74,601 events  |  mag range 4.5 to 8.8


,time,magnitude
0,2016-04-10 00:30:19.840000+00:00,4.5
1,2016-04-10 02:14:34.590000+00:00,5.7
2,2016-04-10 04:22:00.930000+00:00,4.9
3,2016-04-10 06:40:51.830000+00:00,4.7
4,2016-04-10 07:11:21.580000+00:00,5.5


In [11]:
events["time"] = pd.to_datetime(events["time"], utc=True)
events["timestamp"] = events["time"].astype(np.int64) // 10**9
events = events[["timestamp", "magnitude"]]
events.to_parquet("events.parquet", index=False)